In [ ]:
from matplotlib import pyplot as plt
from scipy.stats import norm
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import lognorm
import os

directory = "/users/cdcook/VSP/graphics/"

sns.set_theme(style="darkgrid")

In [ ]:
with open('/users/cdcook/VSP/datafiles/PSdata/xtetrans_1b_Exp4Final.csv', newline='') as w:
    data = list(csv.reader(w))
    data.pop(0)

In [ ]:
def colorSub(data, max):
    counter = 0
    RotseMag = []
    gPSFmag = []
    rPSFmag = []
    iPSFmag = []
    zPSFmag = []
    yPSFmag = []
    gKRONmag = []
    rKRONmag = []
    iKRONmag = []
    zKRONmag = []
    yKRONmag = []
    bitFlags = []
    pseudoBoloMag = []
    PanRotDiff = []
    counter = 0
    for n in range(max):
        if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0:
            counter = counter + 1
            RotseMag.append(float(data[n][4]))
            bitFlags.append(float(data[n][5]))
            gPSFmag.append(float(data[n][6]))
            rPSFmag.append(float(data[n][8]))
            iPSFmag.append(float(data[n][10]))
            zPSFmag.append(float(data[n][12]))
            yPSFmag.append(float(data[n][14]))
            gKRONmag.append(float(data[n][7]))
            rKRONmag.append(float(data[n][9]))
            iKRONmag.append(float(data[n][11]))
            zKRONmag.append(float(data[n][13]))
            yKRONmag.append(float(data[n][15]))
            gflux = pow(10, (float(data[n][6])+48.6)/-2.5)
            rflux = pow(10, (float(data[n][8])+48.6)/-2.5)
            iflux = pow(10, (float(data[n][10])+48.6)/-2.5)
            zflux = pow(10, (float(data[n][12])+48.6)/-2.5)
            yflux = pow(10, (float(data[n][14])+48.6)/-2.5)
            #totalFlux = (gflux*(7.058-5.454) + rflux*(5.454-4.347) + iflux*(4.347-3.68) + zflux*(3.68-3.26) +yflux*(3.26-3))/((7.058-3))
            #totalFlux = (gflux*(52.5) + rflux*(97.5) + iflux*(106.25) + zflux*(72.5) +yflux*(52.5))/((381.25))
            #totalFlux = (gflux*(551-414) + rflux*(689-550) + iflux*(819-690) + zflux*(922-818) +yflux*(1001-918))/((592))
            totalFlux = (gflux*(0.1212) + rflux*(0.1463) + iflux*(0.1435) + zflux*(0.098) +yflux*(.0393))/((0.5483))
            logpart = math.log(totalFlux/3631e-23, 10)
            pseudoBoloMag.append(-2.5*logpart)
            PanRotDiff.append(-2.5*logpart - float(data[n][4]))
    data={'RotseMag':RotseMag, 'bitFlags':bitFlags, 'gPSFmag':gPSFmag, 'rPSFmag':rPSFmag, 'iPSFmag':iPSFmag, 'zPSFmag':zPSFmag, 'yPSFmag':yPSFmag, 'gKRONmag':gKRONmag, 'rKRONmag':rKRONmag, 'iKRONmag':iKRONmag, 'zKRONmag':zKRONmag, 'yKRONmag':yKRONmag, 'pseudoBoloMag':pseudoBoloMag, 'PanRotDiff':PanRotDiff}
    combinedDF = pd.DataFrame(data)
    return combinedDF    

In [ ]:
def KronCut(kronBand, kronDist, dataDF):
    if (kronBand == 'g'):
        kronName = 'gKRONmag'
        psfName = 'gPSFmag'
        kronCutName = 'gKron'
    if (kronBand == 'r'):
        kronName = 'rKRONmag'
        psfName = 'rPSFmag'
        kronCutName = 'rKron'
    if (kronBand == 'i'):
        kronName = 'iKRONmag'
        psfName = 'iPSFmag'
        kronCutName = 'iKron'
    if (kronBand == 'z'):
        kronName = 'zKRONmag'
        psfName = 'zPSFmag'
        kronCutName = 'zKron'
    if (kronBand == 'y'):
        kronName = 'yKRONmag'
        psfName = 'yPSFmag'
        kronCutName = 'yKron'
        
    # Condition 1: psfName - kronName should be less than kronDist
    condition1 = abs(dataDF[psfName] - dataDF[kronName]) > kronDist
    # Apply both conditions to the DataFrame
    dataDF = dataDF[condition1]
    
    return dataDF, kronName, psfName, kronCutName

def BitFlagCut(bitFlags, dataDF):
    dataDF = dataDF[dataDF['bitFlags'] != bitFlags]
    return dataDF

def ColorCut(dataDF, sub1, sub2):
    if (sub1 == 'gr'):
        band1a = 'gPSFmag'
        band1b = 'rPSFmag'
        band1name = "g-r"
        
    elif (sub1 == 'gi'):
        band1a = 'gPSFmag'
        band1b = 'iPSFmag'
        band1name = "g-i"
        
    elif (sub1 == 'ri'):
        band1a = 'rPSFmag'
        band1b = 'iPSFmag'
        band1name = "r-i"
        
    if (sub2 == 'gr'):
        band2a = 'gPSFmag'
        band2b = 'rPSFmag'
        band2name = "g-r"
    
    elif (sub2 == 'gi'):
        band2a = 'gPSFmag'
        band2b = 'iPSFmag'
        band2name = "g-i"
        
    elif (sub2 == 'ri'):
        band2a = 'rPSFmag'
        band2b = 'iPSFmag'
        band2name = "r-i"
        
    x = []
    y = []
    for index, row in dataDF.iterrows():
        x.append(row[band1a] - row[band1b])
        y.append(row[band2a] - row[band2b])

    dataDF.insert(0, 'color1', x)
    dataDF.insert(1, 'color2', y)
    return dataDF, band1a, band1b, band2a, band2b, band1name, band2name

In [ ]:
bitFlags = 2.0
color2 = 'gr'
color1 = 'ri'
kronBand = 'r'
kronDist = '0.5'

kronCutTF = False
bitFlagTF = False
colorCutTF = False


dataDF = colorSub(data, 8090)
dataDF, band1a, band1b, band2a, band2b, band1name, band2name = ColorCut(dataDF, 'gr', 'ri')

In [ ]:
dataDF, kronName, psfName, kronCutName = KronCut(kronBand='r', kronDist = 0.5, dataDF = dataDF)
kronCutTF = True

In [ ]:
dataDF = BitFlagCut(bitFlags = 4.0, dataDF = dataDF)
bitFlagTF = True

In [ ]:
dataDF = dataDF.drop(dataDF[dataDF['color2'] > 1.5].index)
dataDF = dataDF.drop(dataDF[dataDF['color2'] < -0.2].index)
xframe = dataDF['color2']
yframe = dataDF['color1']

fit = np.poly1d(np.polyfit(xframe, yframe, 6))

dataDF = dataDF.drop(dataDF[abs(dataDF['color1'] - fit(dataDF['color2'])) < 0.2].index)
colorCutTF = True

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(x="color2", y="color1", data=dataDF)
plt.xlim(-1,3)
plt.ylim(-1,3)

plt.xlabel(band2name, fontsize=20)
plt.ylabel(band1name, fontsize=20)

text = "Fail Color Locus Plot"
fileName = "Fail_CL"
if(kronCutTF):
    text = text + " || " + kronCutName + " +-" + kronDist
    fileName = fileName + "_" + kronCutName + kronDist
if(bitFlagTF):
    text = text + " || " + "E-flag cut"
    fileName = fileName + "_flagCut"
if(colorCutTF):
    text = text + " || " + "Color Cut"
    fileName = fileName + "_colorCut"

fileName = fileName + ".png"

#text = "Saturated in Rotse || rKron+-0.5"
#fileName = "CL_rKron0.5_Sat_Rotse.png"

plt.title(text, fontsize=20)
plt.savefig(directory + fileName)
plt.show()

In [ ]:
params = np.polyfit(dataDF['RotseMag'], dataDF['pseudoBoloMag'], 1, full=False, cov=True)   
x = np.linspace(8, 20, 10000)   
print("slope: ", params[0][0])
print("AB offset: ", params[0][1])
print("Total Count: ", dataDF.shape[0])
plt.figure(figsize=(18,9))
ax = sns.scatterplot(x='RotseMag', y='pseudoBoloMag', data=dataDF)
plt.plot(x, vsp.oneDFit(params[0][0], params[0][1], x), 'r--')
#plt.plot(x, oneDFit(params[0][0], params[0][1], x) + 0.8, 'r--')
plt.xlim(8,20)
plt.ylim(8,22)
plt.xlabel('ROTSE Mag', fontsize=20)
plt.ylabel('Pan Mag', fontsize=20)
ax.text(9,18,"slope: %4.4f" % params[0][0], fontsize=20)
ax.text(9,17,"AB offset: %4.4f" % params[0][1], fontsize=20)
ax.text(9,16,"Total Count: %s" % dataDF.shape[0], fontsize=20)

text = "Fail || Psuedo-Bolometric zero point"
fileName = "Fail_PB_zero_point"
if(kronCutTF):
    text = text + " || " + kronCutName + " +-" + kronDist
    fileName = fileName + "_" + kronCutName + kronDist
if(bitFlagTF):
    text = text + " || " + "E-flag cut"
    fileName = fileName + "_flagCut"
if(colorCutTF):
    text = text + " || " + "Color Cut"
    fileName = fileName + "_colorCut"

fileName = fileName + ".png"

#text = "Saturated in Rotse || gKron+-0.5"
#fileName = "PB_gKron0.5_Sat_Rotse.png"

plt.title(text, fontsize = 20)

plt.savefig(directory + fileName)
plt.show()

In [ ]:
def lognormFIT(x, sigma, loc , scale):
    return scale*lognorm.pdf(x, s=sigma, loc=loc)
    
def doubleGaussian(x, c1, mu1, sigma1, c2, mu2, sigma2):
    return c1*np.exp(-((x-mu1)**2)/(2*sigma1**2)) + c2*np.exp(-((x-mu2)**2)/(2*sigma2**2))

def Gaussian(x, c, mu, sigma):
    return c*np.exp(-((x-mu)**2)/(2*sigma**2))

plt.figure(figsize=(18,9))
ax = sns.histplot(x=dataDF['PanRotDiff'], color="b", bins=250, stat='probability')
xlist = []
ylist = []
for i in ax.patches:
    if (i.get_x() > -0.8 and i.get_x() < 0.6):
        xlist.append(i.get_x())
        ylist.append(i.get_height())

popt, pcov = curve_fit(lognormFIT, xlist, ylist, p0=[0.15, -1, 100], maxfev=100000, bounds=([-5, -5, -100], [1, 1, 100]))
print(popt)

popt2, pcov2 = curve_fit(doubleGaussian, xlist, ylist, p0=[1, -1, 0.5, 1, -1, 0.5], maxfev=100000, bounds=([-100, -2, 0, -100, -2, 0], [1, 1, 100, 1, 1, 100]))

popt3, pcov3 = curve_fit(Gaussian, xlist, ylist, p0=[1, -1, 0.5], maxfev=100000, bounds=([-100, -2, 0], [100, 1, 1]))

yobv = ylist
yexpDG = doubleGaussian(xlist, popt2[0], popt2[1], popt2[2], popt2[3], popt2[4], popt2[5])
yexpLN = lognormFIT(xlist, popt[0], popt[1], popt[2])
yexpGA = Gaussian(xlist, popt3[0], popt3[1], popt3[2])
    
chisqLN = 0
for n in range(yobv.__len__()):
    if(xlist[n] > -0.5 and xlist[n] < 0.25):
        temp = ((yobv[n] - yexpLN[n])**2)/yexpLN[n]
        #print(temp)
        chisqLN = chisqLN + temp
    
chisqDG = 0
for n in range(yobv.__len__()):
    if(xlist[n] > -0.5 and xlist[n] < 0.25):
        temp = chisqDG + ((yobv[n] - yexpDG[n])**2)/yexpDG[n]
        #print(temp)
        chisqDG = chisqDG + temp

chisqGA = 0
for n in range(yobv.__len__()):
    if(xlist[n] > -0.5 and xlist[n] < 0.25):
        temp = chisqGA + ((yobv[n] - yexpGA[n])**2)/yexpGA[n]
        #print(temp)
        chisqGA = chisqGA + temp

print("Chi Square (Double Gaussian): ", chisqDG)
print("Chi Square (Lognormal): ", chisqLN)
print("Chi Sqaure (Gaussian): ", chisqGA)

p = lognorm.pdf(xlist, 1, 0, 1)

#plt.text(1,1,"Chi Square (Double Gaussian): %4.4f" % chisqDG)
#plt.text(1,1,"Chi Square (Lognormal): %4.4f" % chisqLN)
#plt.text(1,1,"Chi Square (Gaussian): %4.4f" % chisqGA)

plt.plot(xlist, lognormFIT(xlist, popt[0], popt[1], popt[2]), 'r--', label='Lognormal fit')
plt.plot(xlist, doubleGaussian(xlist, popt2[0], popt2[1], popt2[2], popt2[3], popt2[4], popt2[5]), 'k--', label='Double Gaussian Fit')
plt.plot(xlist, Gaussian(xlist, popt3[0], popt3[1], popt3[2]), 'p--', label='Gaussian Fit')
plt.xlim(-1,1)
plt.legend(labels=[
    "Lognormal fit: sigma = " + str.format('{0:.4f}', popt[0]) + ", loc = " + str.format('{0:.4f}', popt[1]) + ", scale = " + str.format('{0:.4f}', popt[2]),
    "Double Gaussian fit: c1 = " + str.format('{0:.4f}', popt2[0]) + ", mu1 = " + str.format('{0:.4f}', popt2[1]) + ", sigma1 = " + str.format('{0:.4f}', popt2[2]) + ", c2 = " + str.format('{0:.4f}', popt2[3]) + ", mu2 = " + str.format('{0:.4f}', popt2[4]) + ", sigma2 = " + str.format('{0:.4f}', popt2[5]),
    "Gaussian fit: c = " + str.format('{0:.4f}', popt3[0]) + ", mu = " + str.format('{0:.4f}', popt3[1]) + ", sigma = " + str.format('{0:.4f}', popt3[2])
])
plt.xlabel("diff(PanSTARRs, ROTSE)",  fontsize=20)
plt.xlabel("diff(PanSTARRs, ROTSE)",  fontsize=20)
plt.ylabel("Probability",  fontsize=20)

text = "Fail_diff(PanSTARRs, Rotse) Histogram"
fileName = "Fail_diffHist"
if(kronCutTF):
    text = text + " || " + kronCutName + " +-" + kronDist
    fileName = fileName + "_" + kronCutName + kronDist

if(bitFlagTF):
    text = text + " || " + "E-flag cut"
    fileName = fileName + "_flagCut"

if(colorCutTF):
    text = text + " || " + "Color Cut"
    fileName = fileName + "_colorCut"

fileName = fileName + ".png"

#text = "Saturated in Rotse || gKron+-0.5"
#fileName = "Hist_gKron0.5_Sat_Rotse.png"

plt.title(text, fontsize = 20)
plt.savefig(directory + fileName)
plt.show()